# All-atom PhantomWalk: initializing a dense polymer melt

This tutorial builds a dense all-atom polyethylene melt at its target density in under a minute on a CPU:

1. **Build chains** with a `Polymer` (here `PolyEthylene`).
2. **Place them at the target density** with `AtomisticRandomWalk`, turning every rotatable bond with `LocalBoltzmannSampler`. Chains overlap; that is intended.
3. **Write the all-atom DPD force field** with `UFF_DPD`: UFF bonded terms scaled up, plus a soft DPD pair force that lets atoms pass through each other.
4. **Relax** with `PhantomWalk.run_initialization`: DPD until the energies are stationary, then FIRE.
5. **Hand off** the coordinates to a standard force field for minimization.

The result is *minimizer-ready coordinates at the right density*, not an equilibrated melt. Units are kJ/mol, nm and amu. Requires RDKit and OpenFF (openff-toolkit and openff-interchange).

In [ ]:
import warnings

import numpy as np
import unyt as u

from flowermd.library import (
    UFF_DPD,
    AtomisticRandomWalk,
    LocalBoltzmannSampler,
    PhantomWalk,
    PolyEthylene,
)

warnings.filterwarnings("ignore")

## 1-2. Chains placed at the target density

Ten 20-mer chains (about 1,200 atoms). `AtomisticRandomWalk` sizes a cubic box for 0.78 g/cm³ (a polyethylene melt at 413 K) and gives each chain a random start and orientation. `LocalBoltzmannSampler` first relaxes the chain under the Sage force field's bonded terms, then turns every rotatable C-C bond to a torsion drawn from local Boltzmann weights at 413 K: Sage's torsion about the bond plus the Lennard-Jones energy of atoms up to four bonds apart, with no heavy atoms closer than 0.30 nm. This gives realistic chain dimensions; bond lengths, bond angles and any stereocenters keep the relaxed geometry.

In [ ]:
chains = PolyEthylene(lengths=20, num_mols=10)
system = AtomisticRandomWalk(
    chains,
    density=0.78 * u.g / u.cm**3,
    torsion_sampler=LocalBoltzmannSampler(temperature=413),
    seed=1,
)
print("atoms:", system.n_particles, " box (nm):", system.box.Lx)

## 3. The all-atom DPD force field

`UFF_DPD` writes a GMSO XML force field for this system and renames its particles to the refined atom types. Defaults follow the PhantomWalk protocol: UFF bonded terms ×30 (torsions too), DPD repulsion `A = 52300` kJ/(mol nm) and friction `gamma = 4091` amu/ps, each weighted per pair by the UFF Lennard-Jones well depths, and a 0.35 nm cutoff. Applying a DPD force field also restrains every tetrahedral stereocenter, so DPD cannot invert it (`AtomisticRandomWalk(protect_stereo=False)` turns this off); polyethylene has none.

In [ ]:
ff = UFF_DPD(system, file_path="pe_dpd.xml")
system.apply_forcefield(r_cut=0.35, force_field=ff, kT=4.184)
print("types:", system.hoomd_snapshot.particles.types)

## 4. DPD until stationary, then FIRE

`run_initialization` runs 3,500 DPD steps, then 500-step chunks (five energy samples each) until every force's per-particle energy changes by less than 2 % for two chunks in a row, capped at `n_steps_dpd`. FIRE then runs with the conservative part of the DPD force. If the cap is reached the coordinates are still returned with `dpd_converged = False` and a warning (`require_convergence=True` makes it an error).

In [ ]:
sim = PhantomWalk(
    system.hoomd_snapshot,
    system.hoomd_forcefield,
    n_steps_dpd=40000,
    n_steps_fire=100,
    dt=2e-4,
    gsd_write_freq=1e5,
    log_write_freq=1e5,
    run_now=False,
)
record = sim.run_initialization(record_file="pe_record.json")
print("DPD steps:", record["dpd_steps"], " converged:", record["dpd_converged"])
print("ring piercings:", record["ring_piercings"])
print(np.array(record["dpd_energy_history"])[-3:].round(3))

## 5. Hand-off

The final unwrapped coordinates (nm) go back into the mBuild compound for a parameterizer such as OpenFF Interchange or foyer; `sim.save_restart_gsd()` keeps the HOOMD state instead.

In [ ]:
snap = sim.state.get_snapshot()
system.system.xyz = (
    snap.particles.position + snap.particles.image * snap.configuration.box[:3]
)
system.system.save("pe_melt.mol2", overwrite=True)